<a href="https://colab.research.google.com/github/DhimanTarafdar/next-word-prediction-using-LSTM/blob/main/LSTM_project_code_explanation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LSTM Project — Complex Code Explanation Notes

---

## 1. Vocabulary Building — `Counter` দিয়ে Vocab তৈরি

```python
from collections import Counter

vocab = { '<unk>' : 0 }

for token in Counter(tokens).keys():
    if token not in vocab:
        vocab[token] = len(vocab)
```

**Explanation:**

`Counter(tokens)` হলো একটি Python built-in tool যেটা একটা list-এর প্রতিটা element কতবার আছে সেটা count করে। এখানে `.keys()` দিয়ে শুধু unique words গুলো বের করা হচ্ছে — frequency দরকার নেই, শুধু কোন কোন word আছে সেটা দরকার।

`vocab` হলো একটা **Dictionary** যেখানে প্রতিটা word-এর একটা unique **integer index** assign করা হচ্ছে।

- `'<unk>'` মানে **Unknown token** — যে word vocab-এ নেই সেটাকে এই index দেওয়া হয়।
- `len(vocab)` use করা হচ্ছে কারণ প্রতিবার নতুন word যোগ হলে vocab-এর size বাড়ে, তাই index automatically increment হয়।
**Example:**
```
tokens = ["the", "cat", "sat", "the"]
Counter(tokens) → {"the": 2, "cat": 1, "sat": 1}
Counter(tokens).keys() → ["the", "cat", "sat"]

vocab = {"<unk>": 0, "the": 1, "cat": 2, "sat": 3}
```

---

## 2. Text to Indices — `text_to_indices()` Function

```python
def text_to_indices(sentence, vocab):
    numerical_sentence = []
    for token in sentence:
        if token in vocab:
            numerical_sentence.append(vocab[token])
        else:
            numerical_sentence.append(vocab['<unk>'])
    return numerical_sentence
```

**Explanation:**

Neural Network text বুঝতে পারে না, সে শুধু **number** বোঝে। তাই এই function প্রতিটা word-কে তার vocab-এ assigned **integer index**-এ convert করে।

- যদি word vocab-এ **থাকে** → সেই word-এর index নাও
- যদি word vocab-এ **না থাকে** → `<unk>`-এর index (0) দাও
**Example:**
```
vocab = {"<unk>": 0, "the": 1, "cat": 2, "sat": 3}
sentence = ["the", "dog", "sat"]

→ [1, 0, 3]
("dog" vocab-এ নেই, তাই <unk> = 0)
```

---

## 3. Training Sequence তৈরি — Sliding Window Logic

```python
training_sequence = []
for sentence in input_numerical_sentence:
    for i in range(1, len(sentence)):
        training_sequence.append(sentence[:i+1])
```

**Explanation:**

এটা LSTM-এর জন্য **next word prediction** training data তৈরির সবচেয়ে গুরুত্বপূর্ণ step।

ধরো sentence হলো: `[10, 25, 47, 62]`

Inner loop যা করে:
- `i=1` → `[10, 25]`
- `i=2` → `[10, 25, 47]`
- `i=3` → `[10, 25, 47, 62]`
এভাবে প্রতিটি sub-sequence তৈরি হয় যেখানে **শেষ word হলো target (y)** এবং তার আগের সব word হলো **input (X)**।

মানে model শেখে: "এই words গুলো দেখলে পরের word কী হবে?"

এটাকে বলা হয় **Sliding Window** বা **n-gram style training**।

---

## 4. Padding — Variable Length Sequences সমান করা

```python
padded_training_sequence = []
for sequence in training_sequence:
    padded_training_sequence.append(
        [0] * (max(len_list) - len(sequence)) + sequence
    )
```

**Explanation:**

LSTM-এ সব sequence-এর **length সমান** থাকতে হয় কারণ PyTorch tensor-এ rectangular matrix দরকার।

যেমন:
- max length = 25
- একটা sequence = `[10, 25, 47]` (length 3)
- Padding করলে → `[0, 0, 0, 0, ...(22 টা শূন্য)..., 10, 25, 47]`
**Left padding** করা হচ্ছে (শূন্যগুলো বাম দিকে বসছে) যাতে **শেষের actual content** সবসময় একই position-এ থাকে। `0` হলো padding index — এটাই `<unk>`-এর index — কিন্তু এখানে এটা শুধু placeholder হিসেবে ব্যবহার হচ্ছে।

---